In [10]:
import numpy as np
import librosa
import os, glob, joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

In [11]:
CANONICAL_SR = 22050
MIN_DURATION_SEC = 0.5  #rejection buffer after trim
N_MFCC = 20

In [12]:
def extractChordFeatures(y, sr, target_sr=CANONICAL_SR):
    # we are extracting a compact, time invariant feature vector for chord recognition....and returning a fixed length 1D array (12+20+20+24 = 76 features)

    #first we need to resample to our set rate
    if sr!= target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    #ensure float32 mono
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = np.mean(y, axis=0)
    
    #then we trim any leading or trailing silence:
    try:
        y_trimmed, _ = librosa.effects.trim(y, top_db=30)
        if len(y_trimmed) > 0:
            y = y_trimmed
    except Exception:
        pass

    #rejecting too short signals
    min_samples = int(MIN_DURATION_SEC * sr)
    if len(y) < min_samples:
        return None

    #this is our harmonic/percussive separation --chords are harmonic
    y_harm, _ = librosa.effects.hpss(y)

    #Our Chroma(this is the chord feature) --mean over time
    chroma = librosa.feature.chroma_cqt(y=y_harm, sr=sr)
    chroma_mean = chroma.mean(axis=1)
    chroma_std = chroma.std(axis=1)

    #MFCCs(timbre) --mean and standard deviation over time
    mfcc = librosa.feature.mfcc(y=y_harm, sr=sr, n_mfcc=N_MFCC)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    #special constrast --mean over time
    contrast = librosa.feature.spectral_contrast(y=y_harm, sr=sr)
    contrast_mean = contrast.mean(axis=1)
    feat = np.concatenate([chroma_mean, chroma_std, mfcc_mean, mfcc_std, contrast_mean])

    return feat.astype(np.float32)

In [13]:
def build_dataset(pattern, verbose=True):
    X, Y = [], []
    dropped = 0
    for f in glob.glob(pattern):
        y, sr = librosa.load(f, sr=None)
        feat = extractChordFeatures(y, sr)
        if feat is None:
            if verbose:
                print(f"dropped (too short/silent): {f}")
            dropped+=1
            continue
        X.append(feat)
        Y.append(os.path.basename(os.path.dirname(f)))
    if verbose:
        print(f"Loaded {len(X)} files, dropped {dropped}")
    return np.array(X), np.array(Y)

In [14]:
X_train, Y_train = build_dataset("./Training/*/*.wav")
X_test, Y_test = build_dataset("./Test/*/*.wav")
le = LabelEncoder()
y_train = le.fit_transform(Y_train)
y_test = le.transform(Y_test)

c:\Users\oyin1\OneDrive - UT Arlington\Documents\Chord Recognition\chord-recognition-env\Lib\site-packages\librosa\core\constantq.py:1202: UserWarning: n_fft=1024 is too large for input signal of length=760
  D = stft(
c:\Users\oyin1\OneDrive - UT Arlington\Documents\Chord Recognition\chord-recognition-env\Lib\site-packages\librosa\core\constantq.py:1202: UserWarning: n_fft=1024 is too large for input signal of length=952
  D = stft(
c:\Users\oyin1\OneDrive - UT Arlington\Documents\Chord Recognition\chord-recognition-env\Lib\site-packages\librosa\core\constantq.py:1202: UserWarning: n_fft=1024 is too large for input signal of length=968
  D = stft(
c:\Users\oyin1\OneDrive - UT Arlington\Documents\Chord Recognition\chord-recognition-env\Lib\site-packages\librosa\core\constantq.py:1202: UserWarning: n_fft=1024 is too large for input signal of length=1008
  D = stft(
c:\Users\oyin1\OneDrive - UT Arlington\Documents\Chord Recognition\chord-recognition-env\Lib\site-packages\librosa\core\con

KeyboardInterrupt: 

In [ ]:
clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=10, gamma="scale", probability=True)
)
clf.fit(X_train, y_train)
print("Train acc:", clf.score(X_train, y_train))
print("Test  acc:", clf.score(X_test,  y_test))
print("CV 5-fold:", cross_val_score(clf, X_train, y_train, cv=5).mean())